this notebook reads the IMC files from different subfolders, it aggregates and uniforms the formatting

In [1]:
import numpy as np
import pandas as pd
from os import listdir
import re
from os.path import isfile, join
import os
import warnings

def strip_extension(self):
    '''strip the file extension from a pd.Series of file names'''
    return self.apply(lambda x: os.path.splitext(x)[0])

# Add the method to the Pandas Series class
pd.Series.strip_extension = strip_extension


# Formatting the intensities

The aim of this section is to aggregate all the data into a unique csv file. I add the ROI as a column

In [2]:
path0 = '../../IMC_SegmentationResults/'
Leap_list = [name for name in next(os.walk(path0))[1] if name[:4]=='Leap']#list of Leap files
#path = path0+'all_data/'
directory = 'intensities/'
#file_names = [f for f in listdir(path+directory) if isfile(join(path+directory, f))]
#stripped_names = [re.sub(r'\.[^.]+$', '', name) for name in file_names]


In [7]:
#[x[0] for x in os.walk(path0)]
def filename_mapper():
    '''for every file in the intensity folder, put in a Leap format'''
    file_names = []
    for  name in ['Leap002', 'Leap001']:
        path = path0+name+'/'+directory
        for f in listdir(path):
            if isfile(join(path, f)):#select only files,  not directories
                #if file is not in the format Leap***_***.csv, create the corresponding name
                pattern = r'^Leap\d{3}_\d{3}\.csv$'
                if re.match(pattern, f) is not None:
                    file_names+=[(f,f.split('.')[0])]#strip the extension from the mapper name
                else:
                    new_name = name+'_'+f.split('_')[-1].split('.')[0]# use the Leap name of the folder
                    file_names+=[(f,new_name)]
    #return file_names
    tb = pd.DataFrame(file_names,columns=['file_name','new_name'])
    tb['folder'] = tb['new_name'].str.split('_').str[0]
    tb['no_ext'] = tb.file_name.strip_extension()
    return tb
filenames = filename_mapper()

In [8]:
Leap_list

['Leap008', 'Leap024', 'Leap003', 'Leap004_2', 'Leap002', 'Leap001', 'Leap004']

In [9]:
print(filenames.shape)
filenames.head()



(9, 4)


,file_name,new_name,folder,no_ext
0,Leap002_002.csv,Leap002_002,Leap002,Leap002_002
1,Leap002_003.csv,Leap002_003,Leap002,Leap002_003
2,Leap002_006.csv,Leap002_006,Leap002,Leap002_006
3,Leap002_007.csv,Leap002_007,Leap002,Leap002_007
4,Leap002_005.csv,Leap002_005,Leap002,Leap002_005


For every intensity file, I associate the appropriate ROI

In [10]:
all_images = pd.DataFrame()
for folder in filenames['folder']:
    images = pd.read_csv(path0+folder+'/images.csv') 
    images['ROI'] = images.acquisition_description.str.extract(r'(ROI_\d+)')#extract ROI from string
    images['ROI'] = images.ROI.str.split('_').str[1]
    all_images = pd.concat((all_images,images),ignore_index=True)
all_images['image_no_ext'] = all_images.image.strip_extension()
all_images.drop_duplicates(inplace=True)
merge = pd.merge(left = filenames,right=all_images,left_on='no_ext',right_on='image_no_ext',how = 'inner')
merge.drop('no_ext',axis =1,inplace=True)#remove one of the column used to merge
merge.shape

(8, 20)

In [11]:
merge.head()


,file_name,new_name,folder,image,width_px,height_px,num_channels,source_file,recovery_file,recovered,acquisition_id,acquisition_description,acquisition_start_x_um,acquisition_start_y_um,acquisition_end_x_um,acquisition_end_y_um,acquisition_width_um,acquisition_height_um,ROI,image_no_ext
0,Leap002_003.csv,Leap002_003,Leap002,Leap002_003.tiff,1014,975,62,Leap002.mcd,Leap002_ROI_007_20001982_x29480_y7823_w1014_h9...,False,3,ROI_007_20001982_x29480_y7823_w1014_h975,29480.0,7823.0,30494.0,6848.0,1014.0,975.0,007,Leap002_003
1,Leap002_006.csv,Leap002_006,Leap002,Leap002_006.tiff,1014,975,62,Leap002.mcd,Leap002_ROI_003_20001982_x33982_y12851_w1014_h...,False,6,ROI_003_20001982_x33982_y12851_w1014_h975,33982.0,12851.0,34996.0,11876.0,1014.0,975.0,003,Leap002_006
2,Leap002_007.csv,Leap002_007,Leap002,Leap002_007.tiff,1014,1077,62,Leap002.mcd,Leap002_ROI_002_20001982_x33707_y14202_w1014_h...,False,7,ROI_002_20001982_x33707_y14202_w1014_h1077,33707.0,14202.0,34721.0,13125.0,1014.0,1077.0,002,Leap002_007
3,Leap002_005.csv,Leap002_005,Leap002,Leap002_005.tiff,1014,975,62,Leap002.mcd,Leap002_ROI_004_20001982_x26610_y10850_w1014_h...,False,5,ROI_004_20001982_x26610_y10850_w1014_h975,26610.0,10850.0,27624.0,9875.0,1014.0,975.0,004,Leap002_005
4,Leap002_004.csv,Leap002_004,Leap002,Leap002_004.tiff,1014,975,62,Leap002.mcd,Leap002_ROI_005_20001982_x30202_y11278_w1014_h...,False,4,ROI_005_20001982_x30202_y11278_w1014_h975,30202.0,11278.0,31216.0,10303.0,1014.0,975.0,005,Leap002_004


For every cell, I add the a column indicating the slide identifier

In [12]:
merged_intensities = pd.DataFrame()
merged_regions = pd.DataFrame()
for _,row  in merge.iterrows():
    #concatente together the intensities files
    df = pd.read_csv(path0+'/'+row.folder+'/intensities/'+row.file_name)
    df = df[df.columns[df.columns.str.find('-')>0]]#select only columns related to antibody and filter out empty channels
    df.columns = df.columns.str.replace(r'^.*?-','',regex = True)#remove the metals
    df['slice_ID'] = row.new_name
    merged_intensities = pd.concat([merged_intensities,df],ignore_index=True)

    #Concatenate together the region files which contains the spatial information for every cell
    region_file = path0+'/'+row.folder+'/regionprops/'+row.file_name
    if os.path.isfile(region_file):
        df_regions = pd.read_csv(region_file)
        df_regions['slice_ID'] = row.new_name
        merged_regions = pd.concat([merged_regions,df_regions.iloc[:,1:]],ignore_index=True)#remove the object column
    else:
        warnings.warn('File '+region_file+'  does not exist')
print('shape of the dataframe:',merged_intensities.shape)
merged_intensities.head()

shape of the dataframe: (58031, 38)


,CD38,EGFR,p53,CD14,Tbet,CD16,CD163,Pan-keratin,CD11b,PD-L1,...,CD3,CD27,PD-L2,CD45RO,Alpha-SMA,Vimentin,CD31,DNA1,DNA2,slice_ID
0,0.525571,0.490078,0.764117,0.868680,0.639988,0.177013,0.062500,1.165243,1.428627,0.176200,...,0.687737,0.144994,0.467670,1.169612,1.407416,0.506011,0.125198,4.204468,9.026807,Leap002_003
1,0.515276,0.324182,0.076923,0.305964,0.519610,0.085043,0.000000,1.743893,0.845461,0.370850,...,0.858610,0.308680,0.413358,0.976917,1.002858,0.631692,0.313133,12.053872,18.935056,Leap002_003
2,0.436287,0.675673,0.561728,1.066378,0.475544,0.408479,0.186198,1.345208,1.613524,0.280647,...,0.967371,0.590248,0.505237,0.686423,0.676052,0.496595,0.438376,9.899534,17.316983,Leap002_003
3,0.471603,1.317043,0.358976,1.175608,0.543862,0.375068,0.177496,3.742146,1.586432,0.531933,...,0.879761,0.549276,0.451247,0.784658,0.819292,0.462176,0.589874,7.838971,12.750305,Leap002_003
4,0.694710,0.684521,0.289157,1.725331,0.676068,0.697254,0.130107,1.264105,2.430705,0.228462,...,1.211621,0.504643,0.218492,1.419250,0.382099,0.308627,0.245315,4.029468,5.821431,Leap002_003


In [16]:
merged_intensities

,CD38,EGFR,p53,CD14,Tbet,CD16,CD163,Pan-keratin,CD11b,PD-L1,...,CD3,CD27,PD-L2,CD45RO,Alpha-SMA,Vimentin,CD31,DNA1,DNA2,slice_ID
0,0.086607,0.035714,0.035714,0.080756,0.017857,0.057445,0.017857,0.000000,0.035714,0.064036,...,0.000000,0.000000,0.017857,0.055358,0.425580,0.193092,0.042434,0.536611,0.987804,Leap031_006
1,0.175462,0.302845,0.238049,0.349944,0.111111,0.265496,0.227717,0.055556,0.275023,0.309092,...,0.173632,0.055556,0.055556,0.387039,0.248602,0.634744,0.501567,1.327157,3.673505,Leap031_006
2,0.763631,0.181818,0.000000,0.931856,0.096269,0.996706,1.003603,0.131227,0.000000,0.000000,...,0.198596,0.000000,0.000000,0.681822,0.000000,0.000000,0.000000,0.845051,1.715105,Leap031_006
3,0.123132,0.221118,0.195240,0.371842,0.082601,0.266941,1.133270,0.034856,0.202127,0.137550,...,0.977854,0.055803,0.140183,0.403229,0.075408,1.264093,0.096260,2.041517,3.446062,Leap031_006
4,0.158077,0.210496,0.060787,0.350709,0.038462,0.133366,2.212801,0.131235,0.076923,0.103961,...,0.076923,0.000000,0.124069,0.178207,0.000000,0.159360,0.000000,1.440386,2.084120,Leap031_006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198515,0.418936,0.331470,0.084305,0.358807,0.231275,0.206789,0.125666,0.000000,0.307107,0.156502,...,0.058824,0.507038,0.029412,0.160392,0.644993,7.584033,0.635834,2.815906,5.621788,Leap002_003
198516,0.415011,0.281478,0.273077,0.703713,0.119866,0.286251,2.428183,0.043478,0.338618,0.339245,...,0.454446,0.937914,0.364163,0.904218,0.569790,14.500667,0.984928,9.310328,15.991977,Leap002_003
198517,0.000000,0.166667,0.166667,0.333333,0.383504,0.166667,0.368397,0.166667,0.135580,0.861937,...,0.000000,0.500000,0.000000,0.500000,0.333333,5.530557,0.000000,1.041990,2.094379,Leap002_003
198518,0.353855,0.291492,0.277962,0.586611,0.182899,0.145833,0.134002,0.199694,0.325813,0.304776,...,0.317032,0.259739,0.150173,1.207261,0.760741,15.218093,0.397587,4.858346,8.054574,Leap002_003


save the files

In [10]:
cond = merged_intensities.slice_ID.str.split('_').str[0].isin(['Leap001','Leap002'])

,CD38,EGFR,p53,CD14,Tbet,CD16,CD163,Pan-keratin,CD11b,PD-L1,...,CD3,CD27,PD-L2,CD45RO,Alpha-SMA,Vimentin,CD31,DNA1,DNA2,slice_ID
140489,1.461853,0.668968,0.846283,5.433715,0.605918,1.394112,0.457791,0.501216,4.282768,0.871965,...,1.020448,0.481673,0.872981,6.323478,2.815302,26.174426,0.913648,25.471886,44.068921,Leap001_010
140490,0.719551,0.971956,1.022850,7.243093,0.669302,2.003613,1.519651,1.453888,8.515267,0.800962,...,1.614737,0.825029,0.771524,1.785376,1.655509,102.465507,1.320291,5.819796,7.662745,Leap001_010
140491,1.046450,0.742872,0.631756,4.099441,0.739785,0.879591,0.212293,0.644256,5.143122,0.651731,...,1.575910,0.645652,0.584983,5.687128,6.275195,64.646987,2.619213,21.093993,37.990815,Leap001_010
140492,1.679968,0.959730,0.606500,4.311874,0.770750,0.435136,0.265197,0.471985,4.966139,0.789661,...,1.358013,0.817333,0.884363,4.092973,125.630067,35.425157,3.238973,23.544535,40.385064,Leap001_010
140493,1.229798,0.651494,0.670303,10.609113,0.821962,1.357689,0.914957,0.371176,8.346500,0.834000,...,2.887295,1.090713,1.010408,5.049454,68.976499,77.434558,2.880486,25.455198,45.001410,Leap001_010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198515,0.418936,0.331470,0.084305,0.358807,0.231275,0.206789,0.125666,0.000000,0.307107,0.156502,...,0.058824,0.507038,0.029412,0.160392,0.644993,7.584033,0.635834,2.815906,5.621788,Leap002_003
198516,0.415011,0.281478,0.273077,0.703713,0.119866,0.286251,2.428183,0.043478,0.338618,0.339245,...,0.454446,0.937914,0.364163,0.904218,0.569790,14.500667,0.984928,9.310328,15.991977,Leap002_003
198517,0.000000,0.166667,0.166667,0.333333,0.383504,0.166667,0.368397,0.166667,0.135580,0.861937,...,0.000000,0.500000,0.000000,0.500000,0.333333,5.530557,0.000000,1.041990,2.094379,Leap002_003
198518,0.353855,0.291492,0.277962,0.586611,0.182899,0.145833,0.134002,0.199694,0.325813,0.304776,...,0.317032,0.259739,0.150173,1.207261,0.760741,15.218093,0.397587,4.858346,8.054574,Leap002_003


In [11]:
path2 = './pre_processed_files'
isExist = os.path.exists(path2)
if not isExist:

   # Create a new directory because it does not exist
   os.makedirs(path2)
cond = merged_intensities.slice_ID.str.split('_').str[0].isin(['Leap001','Leap002'])
merged_intensities = merged_intensities[cond]
    
merged_intensities.to_csv(path2+'/all_data_intensities.csv')

In [10]:

merged_regions.to_csv('./pre_processed_files/all_data_regions.csv',index=False)

# Creating the signature
I format the signature file to work with AnnoSpat

In [28]:
#signature = pd.read_csv('../IMC_SegmentationResults/IMC_SegmentationResults/IMCCelltypeResults/cell_type_matrix.csv').set_index('Marker')
signature = pd.read_csv(path0+'/IMCCelltypeResults/cell_type_matrix.csv').set_index('Marker')
signature.head()
signature.drop(labels=['Proliferative cells?','Other cells','p53+ cells?'],axis='columns',inplace=True)# drop the proliferative cell class
signature.head()

,Cancer,CD163p_Mac [M2],CD163n_Mac [M1],Naive CD4 T cells,Memory T cells,Regulatory T cells,Naive CD8 T cells,Memory CD8 T cells,Naive B cells,Memory B cells,NK and CD8 Tcell,CI_Monocyte,Int_Monocyte,nonCI_Monocyte,Endothelial,Fibroblasts,B7H4+ cancer cells,Neutrophil&monocyte
Marker,,,,,,,,,,,,,,,,,,
CD38,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EGFR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
p53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CD14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,-1.0,-1.0,NaN,NaN,-1.0
Tbet,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
new = {}
for column in signature.columns:
    protein_2_add = signature.index[~signature[column].isna()]#series of protein to use as markers
    order = np.argsort(signature[~signature[column].isna()][column])[::-1]
    protein_2_add = protein_2_add[order]
    a =signature[column][protein_2_add]
    new[column] = list(protein_2_add.where(a>0,protein_2_add.astype(str)+'-'))
signature =pd.DataFrame.from_dict(new,orient = 'index').T

In [31]:
signature.to_csv('pre_processed_files/signature.csv',index=False)    

In [30]:
signature

,Cancer,CD163p_Mac [M2],CD163n_Mac [M1],Naive CD4 T cells,Memory T cells,Regulatory T cells,Naive CD8 T cells,Memory CD8 T cells,Naive B cells,Memory B cells,NK and CD8 Tcell,CI_Monocyte,Int_Monocyte,nonCI_Monocyte,Endothelial,Fibroblasts,B7H4+ cancer cells,Neutrophil&monocyte
0,Beta-Catenin,CD68,CD68,CD27,CD45RO,CD45RO,CD3,CD45RO,CD20,CD27,Granzyme-B,CD14,CD16,CD16,CD31,Vimentin,B7-H4,CD45
1,E-Cadherin,CD163,Pan-keratin-,CD3,CD27,CD27,CD8a,CD27,CD45,CD20,CD107a,CD3-,CD14,CD3-,CD3-,Alpha-SMA,Beta-Catenin,CD11b
2,Pan-keratin,Pan-keratin-,CD163-,CD4,CD3,CD3,CD45,CD3,CD38,CD45,CD38,CD20-,CD3-,CD20-,CD20-,CD31-,E-Cadherin,CD16
3,None,None,None,CD45,CD4,CD4,CD38,CD8a,CD27-,CD38,Pan-keratin-,CD68-,CD20-,CD68-,CD68-,CD45-,Pan-keratin,CD14-
4,None,None,None,CD38,CD44,FOXP3,CD45RO-,CD44,CD3-,CD3-,None,Pan-keratin-,CD68-,Pan-keratin-,Pan-keratin-,Pan-keratin-,None,None
5,None,None,None,CD45RO-,CD45,CD44,CD27-,CD45,FOXP3-,FOXP3-,None,CD16-,Pan-keratin-,CD14-,CD16-,None,None,None
6,None,None,None,CD8a-,CD38,CD45,CD20-,CD38,CD11b-,CD11b-,None,None,None,None,CD14-,None,None,None
7,None,None,None,CD20-,CD8a-,CD38,CD4-,CD20-,None,None,None,None,None,None,None,None,None,None
8,None,None,None,FOXP3-,CD20-,CD8a-,FOXP3-,CD4-,None,None,None,None,None,None,None,None,None,None
9,None,None,None,CD44-,FOXP3-,CD20-,CD44-,FOXP3-,None,None,None,None,None,None,None,None,None,None


In [7]:
signature

,Cancer,CD163p_Mac [M2],CD163n_Mac [M1],Naive CD4 T cells,Memory T cells,Regulatory T cells,Naive CD8 T cells,Memory CD8 T cells,Naive B cells,Memory B cells,NK and CD8 Tcell,CI_Monocyte,Int_Monocyte,nonCI_Monocyte,Endothelial,Fibroblasts,B7H4+ cancer cells,Neutrophil&monocyte
0,Pan-keratin,CD163,CD163-,CD38,CD38,CD38,CD38,CD38,CD38,CD38,CD38,CD14,CD14,CD14-,CD14-,Pan-keratin-,Pan-keratin,CD14-
1,E-Cadherin,Pan-keratin-,Pan-keratin-,CD11b-,CD11b-,CD11b-,CD11b-,CD11b-,CD11b-,CD11b-,Pan-keratin-,CD16-,CD16,CD16,CD16-,CD45-,E-Cadherin,CD16
2,Beta-Catenin,CD68,CD68,CD45,CD45,CD45,CD45,CD45,CD45,CD45,CD107a,Pan-keratin-,Pan-keratin-,Pan-keratin-,Pan-keratin-,Alpha-SMA,Beta-Catenin,CD11b
3,None,None,None,CD44-,CD44,CD44,CD44-,CD44,FOXP3-,FOXP3-,Granzyme-B,CD68-,CD68-,CD68-,CD68-,Vimentin,B7-H4,CD45
4,None,None,None,FOXP3-,FOXP3-,FOXP3,FOXP3-,FOXP3-,CD20,CD20,None,CD20-,CD20-,CD20-,CD20-,CD31-,None,None
5,None,None,None,CD4,CD4,CD4,CD4-,CD4-,CD3-,CD3-,None,CD3-,CD3-,CD3-,CD3-,None,None,None
6,None,None,None,CD20-,CD20-,CD20-,CD20-,CD20-,CD27-,CD27,None,None,None,None,CD31,None,None,None
7,None,None,None,CD8a-,CD8a-,CD8a-,CD8a,CD8a,None,None,None,None,None,None,None,None,None,None
8,None,None,None,CD3,CD3,CD3,CD3,CD3,None,None,None,None,None,None,None,None,None,None
9,None,None,None,CD27,CD27,CD27,CD27-,CD27,None,None,None,None,None,None,None,None,None,None


COde below run AnnoSpat to perform cell type annotation

In [12]:
!python AnnoSpat_main/AnnoSpat/run.py -i pre_processed_files/all_data_intensities.csv  -m pre_processed_files/signature.csv  -o ./output -f CD38 -l DNA2 -r slice_ID -a '[99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5,99.5]'


=====Function chosen: ANNOTATION=====

Starting cell type annotation.....

-----Estimating cell types using Semi supervised clustering-----

-------------------------Done-------------------------

Time taken: 0.019012804826100668seconds

Estimating cell types...
-----Estimating cell types using ELM-----

-------------------------Done-------------------------

Time taken: 0.10888853867848715seconds

Saving Annotations in: /home/giuseppe/Downloads/Annospat_labelling/output/trte_labels_numericLabels_ELM_IMC_T1D_AnnoSpat.csv
-------------------------Done-------------------------



In [21]:
pd.__version__

'2.0.3'